# Lab 02 — 玩家數位指紋的五個維度

**課程**：player-behavior-analytics（玩家行為分析）／第 2 課 數據架構與數位指紋
**目的**：把逐局下注紀錄彙總成每位玩家的「數位指紋」——下注行為、時間、心理（決策節奏）、社交延伸、忠誠度五個維度。
**方式**：全程以 Gemini（AI 助手）產生程式碼——把每個任務的自然語言描述貼給 Gemini，再把生成的程式碼貼進下方 code cell 執行；**重點是觀察結果**，不是寫程式。每個任務下方附參考程式碼，可先自行生成再比對。

> 執行：在 Google Colab 上傳本 .ipynb（或直接開啟），Runtime → Run all。本範本內建模擬數據產生器，無需上傳檔案。

## 0. 載入數據

執行下方 cell 產生模擬數據（同 Lab 01）。本 Lab 同樣隱藏原型欄位（第 4 課揭露）。

維度對照：模擬數據覆蓋「下注流程（Bet Flow）」原始維度；牌路節奏、側注、籌碼軌跡、交易推論等其他原始維度在真實系統中與下注紀錄整合——指紋維度即由這些原始維度彙總而成。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    """模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    """
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df

df = gen_baccarat(seed=42)
df = df.drop(columns=["archetype"])   # 教學簡化：先不看真實原型
df.head()


## 任務 1 — 下注行為維度

在 Gemini 輸入：

> 「按玩家分組，計算每位玩家：最常下注類型、平均下注、最大下注、下注金額變異係數（標準差／平均）、總下注金額。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
fingerprint = df.groupby("player_id").agg(
    top_bet_type=("bet_type", lambda s: s.mode()[0]),
    avg_bet=("bet_amount", "mean"),
    max_bet=("bet_amount", "max"),
    cv_bet=("bet_amount", lambda s: s.std() / s.mean()),
    total_bet=("bet_amount", "sum"),
)
fingerprint.round(2)

**觀察**：cv_bet（變異係數）開始把玩家分開——有人 0.00（固定金額），有人接近 0.5（金額大幅擺動）。

## 任務 2 — 時間維度

在 Gemini 輸入：

> 「按玩家分組，找出每位玩家最常下注的時段（小時）。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour
fingerprint["active_hour"] = (df.groupby("player_id")["hour"]
                                .agg(lambda s: s.mode()[0]))
fingerprint.round(2)

**觀察**：不同玩家活躍時段不同（本例 P01/P06 為 20–22 時、P03/P08 為 18 時、P00 為 14 時）——真實系統中，時段偏好支撐「何時投放優惠」的決策。

## 任務 3 — 心理維度（決策節奏）

在 Gemini 輸入：

> 「對每位玩家：輸了一局之後，下一局加注（超過自己平均下注）的比例是多少？贏了一局之後加注的比例是多少？」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
d = df.sort_values(["player_id", "timestamp"]).copy()
d["prev_win"] = d.groupby("player_id")["is_win"].shift()   # 前一局是否贏
d["avg_bet"] = d.groupby("player_id")["bet_amount"].transform("mean")
d["raised"] = d["bet_amount"] > d["avg_bet"]

rows = []
for pid, g in d.groupby("player_id"):
    rows.append({"player_id": pid,
                 "post_loss_raise": g.loc[g["prev_win"] == False, "raised"].mean(),
                 "post_win_raise": g.loc[g["prev_win"] == True, "raised"].mean()})
rhythm = pd.DataFrame(rows).set_index("player_id")
fingerprint = fingerprint.join(rhythm)
fingerprint.round(2)

**觀察**（本例）：
- post_loss 高、post_win 低＝輸後追注型（如 P01：輸了加注、贏了回到本注）
- post_win 高、post_loss 低＝贏後順勢加注型（如 P02）
- 兩者都低＝紀律固定型（如 P00）
- 兩者都接近 0.5＝金額隨機（波動型，如 P04）

## 任務 4 — 忠誠度／價值維度

在 Gemini 輸入：

> 「按玩家分組，計算：活躍天數、session 數、淨貢獻（總派彩減總下注，即玩家對營收的淨貢獻）。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
loyalty = (df.assign(net=lambda x: x["payout"] - x["bet_amount"])
             .groupby("player_id")
             .agg(days_active=("timestamp", lambda s: pd.to_datetime(s).dt.date.nunique()),
                  sessions=("session_id", "nunique"),
                  net_contribution=("net", "sum")))
fingerprint = fingerprint.join(loyalty)
fingerprint.round(2)

**觀察**：淨貢獻把「貢獻」與「行為」連結——同樣輸贏的兩位玩家，可能來自完全不同的行為模式。

## 任務 5 — 社交延伸維度（說明）

第五個維度（好友群、賭團、共享預算）在真實系統中來自玩家的社交功能與設定檔——模擬數據不含此維度，指紋表中此維度空缺屬預期。營運上它支撐「社交型玩家的團體服務」決策。

## 彙整：完整指紋表

執行下方 cell 查看五維度指紋（社交維度空缺）。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
fingerprint.round(2)

**對照與討論**：
- 指紋表能自動區分幾種行為節奏？（固定型、輸後追注型、贏後順勢型、隨機波動型）
- 哪些欄位鑑別力最強？（cv_bet、post_loss_raise、post_win_raise）
- 注意：有些行為在金額維度上幾乎無法分辨（例如固定金額型 vs 「相信平衡定律」的謬誤型）——第 4 課將用序列特徵（下注變換結構）補足。